In [1]:
# Один раз за сессию. После установки перезапусти сеанс (Среда выполнения → Перезапустить сеанс).
# flash-linear-attention нужен Qwen3.5: без него она считает в 5–10 раз медленнее.
%pip -q install -U  transformers accelerate bitsandbytes xgrammar flash-linear-attention huggingface-hub

In [2]:
!pip list

Package                               Version
------------------------------------- -------------------
absl-py                               1.4.0
accelerate                            1.15.0
access                                1.1.10.post3
affine                                3.0.1
aiofiles                              25.1.0
aiohappyeyeballs                      2.7.1
aiohttp                               3.14.3
aiosignal                             1.4.0
aiosqlite                             0.22.1
alabaster                             1.0.0
albucore                              0.0.24
albumentations                        2.0.8
ale-py                                0.12.1
altair                                5.5.0
annotated-doc                         0.0.5
annotated-types                       0.8.0
antlr4-python3-runtime                4.9.3
anyio                                 4.15.1
anywidget                             0.9.21
apache-tvm-ffi                        0.1.14.

In [ ]:
# токены — из секретов Colab (ключи HF_TOKEN, TG_TOKEN, TG_CHAT), в ноутбуке их не храним
from google.colab import userdata
def secret(k):
    try:
        return userdata.get(k)
    except Exception:
        return None
TG_BOT_TOKEN, TG_CHAT_ID, HF_ROKEN = secret("TG_TOKEN"), secret("TG_CHAT"), secret("HF_TOKEN")


In [5]:
from huggingface_hub import login
login(token=HF_ROKEN)

In [6]:
import transformers, huggingface_hub
print(transformers.__version__, huggingface_hub.__version__)

5.20.0 2.2.0


In [7]:
"""Проверка: какая доля весов каждой модели на самом деле квантована в 4 бита (nf4).

Запускать в Colab в той же среде, что и основной прогон (transformers 5.19.0, bitsandbytes),
после входа в HF (login), отдельной сессией:
    %run check_quant.py
Модели грузятся по очереди так же, как в 07_run_model.ipynb, генерации нет.
bitsandbytes квантует только nn.Linear; эмбеддинги, lm_head и нормализации всегда остаются в bf16,
это нормально. Подозрительна колонка «прочее»: если она в миллиардах, часть слоёв не квантована
(как эксперты MoE у GigaChat в transformers 5).
"""
import gc, json, os, shutil
from collections import defaultdict

import pandas as pd
import torch
import transformers
from huggingface_hub import snapshot_download
from huggingface_hub.constants import HF_HUB_CACHE
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

MODELS = {
    "qwen35_9b":       "Qwen/Qwen3.5-9B",
    "gemma4_e4b":      "google/gemma-4-E4B-it",
    "llama31_8b":      "meta-llama/Llama-3.1-8B-Instruct",
    "ministral3_8b":   "mistralai/Ministral-3-8B-Instruct-2512-BF16",
    "phi4_mini":       "microsoft/Phi-4-mini-instruct",
    # "gigachat31_10b":  "ai-sage/GigaChat3.1-10B-A1.8B-bf16",
    "tlite21_8b":      "t-tech/T-lite-it-2.1",
    "yandex5_lite_8b": "yandex/YandexGPT-5-Lite-8B-instruct",
}
ONLY = None             # список ключей, чтобы проверить не все модели, например ["gemma4_e4b"]
DELETE_CACHE = True     # после проверки удалять веса с диска, иначе на Colab может не хватить места
OUT = "/content/drive/MyDrive/vkr1_results" if os.path.isdir("/content/drive/MyDrive") else "."

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)


def load(key, name):
    """как в ноутбуке: правка config.json у GigaChat и запасной класс для мультимодальных моделей"""
    if key == "gigachat31_10b":
        src = snapshot_download(name)
        fixed = "/content/gigachat_fixed"
        os.makedirs(fixed, exist_ok=True)
        for f in os.listdir(src):
            if f != "config.json" and not os.path.exists(f"{fixed}/{f}"):
                os.symlink(os.path.realpath(f"{src}/{f}"), f"{fixed}/{f}")
        cfg = json.load(open(f"{src}/config.json"))
        cfg["routed_scaling_factor"] = float(cfg["routed_scaling_factor"])
        json.dump(cfg, open(f"{fixed}/config.json", "w"), indent=2)
        name = fixed
    try:
        return AutoModelForCausalLM.from_pretrained(name, quantization_config=bnb, device_map="auto")
    except ValueError:
        from transformers import AutoModelForImageTextToText
        return AutoModelForImageTextToText.from_pretrained(name, quantization_config=bnb, device_map="auto")


def kind(n):
    if "embed" in n:
        return "эмбеддинги"
    if "lm_head" in n:
        return "lm_head"
    if "norm" in n:
        return "нормализации"
    return "прочее"


def inspect(model):
    """миллиарды параметров по группам; 4-битные веса лежат в uint8 по два в байте"""
    size = defaultdict(float)
    other = []
    for n, p in model.named_parameters():       # общие (tied) веса считаются один раз
        if p.dtype == torch.uint8:
            size["квант"] += p.numel() * 2 / 1e9
        else:
            size[kind(n)] += p.numel() / 1e9
            if kind(n) == "прочее":
                other.append((p.numel(), n, str(p.dtype)))
    return size, sorted(other, reverse=True)[:5]


rows = []
for key, name in MODELS.items():
    if ONLY and key not in ONLY:
        continue
    print("\n==", key, name)
    try:
        torch.cuda.reset_peak_memory_stats()
        model = load(key, name)
        size, top = inspect(model)
        total = sum(size.values())
        row = {"model": key, "всего_млрд": round(total, 2), "квант_млрд": round(size["квант"], 2),
               "доля_квант": round(size["квант"] / total, 3),
               **{g: round(size[g], 2) for g in ("эмбеддинги", "lm_head", "нормализации", "прочее")},
               "память_ГБ": round(torch.cuda.memory_allocated() / 1e9, 1), "transformers": transformers.__version__}
        print(row)
        for numel, n, dt in top:                 # самые крупные неквантованные тензоры вне эмбеддингов
            print(f"   {numel / 1e6:9.1f} млн  {dt:15} {n}")
    except Exception as e:
        row = {"model": key, "ошибка": f"{type(e).__name__}: {str(e)[:200]}"}
        print(row)
    rows.append(row)
    model = None
    gc.collect()
    torch.cuda.empty_cache()
    if DELETE_CACHE:
        shutil.rmtree(os.path.join(HF_HUB_CACHE, "models--" + name.replace("/", "--")), ignore_errors=True)
        shutil.rmtree("/content/gigachat_fixed", ignore_errors=True)



== qwen35_9b Qwen/Qwen3.5-9B


config.json:   0%|          | 0.00/3.13k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/79.7k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

{'model': 'yandex5_lite_8b', 'всего_млрд': 8.04, 'квант_млрд': 6.98, 'доля_квант': 0.868, 'эмбеддинги': 0.53, 'lm_head': 0.53, 'нормализации': 0.0, 'прочее': 0.0, 'память_ГБ': 6.1, 'transformers': '5.20.0'}


In [5]:
import transformers, huggingface_hub
print(transformers.__version__, huggingface_hub.__version__)

4.57.6 0.36.2


In [7]:
# GigaChat: проверка квантования в transformers 4.x (эксперты там отдельные nn.Linear, bnb их квантует)
# перед этим один раз: %pip -q install "transformers<5" accelerate bitsandbytes  → перезапустить сеанс
import torch, transformers                                                         
from collections import defaultdict
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

assert transformers.__version__.startswith("4."), "нужен transformers 4.x, перезапусти сеанс после установки"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained("ai-sage/GigaChat3.1-10B-A1.8B-bf16", quantization_config=bnb, device_map="auto")

size = defaultdict(float)
for n, p in model.named_parameters():
    if p.dtype == torch.uint8:
        size["квант"] += p.numel() * 2 / 1e9          # два 4-битных веса в байте
    elif "experts" in n:
        size["эксперты НЕ квант"] += p.numel() / 1e9
    elif "embed" in n or "lm_head" in n:
        size["эмбеддинги + lm_head"] += p.numel() / 1e9
    else:
        size["прочее"] += p.numel() / 1e9
total = sum(size.values())
print("transformers", transformers.__version__)
for k, v in size.items():
    print(f"{k:22} {v:6.2f} млрд  {v / total:6.1%}")
print("память, ГБ:", round(torch.cuda.memory_allocated() / 1e9, 1))

# сохраняем строку GigaChat в тот же файл, что и основная ячейка (колонки эмбеддингов и lm_head тут общие)
import os, pandas as pd
OUT = "/content/drive/MyDrive/vkr1_results" if os.path.isdir("/content/drive/MyDrive") else "."
row = {"model": "gigachat31_10b", "всего_млрд": round(total, 2), "квант_млрд": round(size["квант"], 2),
       "доля_квант": round(size["квант"] / total, 3), "эмбеддинги_и_lm_head": round(size["эмбеддинги + lm_head"], 2),
       "прочее": round(size["прочее"] + size["эксперты НЕ квант"], 2),
       "память_ГБ": round(torch.cuda.memory_allocated() / 1e9, 1), "transformers": transformers.__version__}
path = f"{OUT}/quant_check.csv"
old = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
if len(old):
    old = old[old["model"] != row["model"]]
pd.concat([old, pd.DataFrame([row])], ignore_index=True).to_csv(path, index=False)
print("сохранено:", path)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/4.09G [00:00<?, ?B/s]

...

model-00001-of-00005.safetensors:   0%|          | 0.00/4.09G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Some weights of the model checkpoint at ai-sage/GigaChat3.1-10B-A1.8B-bf16 were not used when initializing DeepseekV3ForCausalLM: ['model.layers.26.eh_proj.weight', 'model.layers.26.embed_tokens.weight', 'model.layers.26.enorm.weight', 'model.layers.26.hnorm.weight', 'model.layers.26.input_layernorm.weight', 'model.layers.26.mlp.experts.0.down_proj.weight', 'model.layers.26.mlp.experts.0.gate_proj.weight', 'model.layers.26.mlp.experts.0.up_proj.weight', 'model.layers.26.mlp.experts.1.down_proj.weight', 'model.layers.26.mlp.experts.1.gate_proj.weight', 'model.layers.26.mlp.experts.1.up_proj.weight', 'model.layers.26.mlp.experts.10.down_proj.weight', 'model.layers.26.mlp.experts.10.gate_proj.weight', 'model.layers.26.mlp.experts.10.up_proj.weight', 'model.layers.26.mlp.experts.11.down_proj.weight', 'model.layers.26.mlp.experts.11.gate_proj.weight', 'model.layers.26.mlp.experts.11.up_proj.weight', 'model.layers.26.mlp.experts.12.down_proj.weight', 'model.layers.26.mlp.experts.12.gate_proj

generation_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

transformers 4.57.6
эмбеддинги + lm_head     0.39 млрд    3.7%
квант                   10.28 млрд   96.3%
прочее                   0.00 млрд    0.0%
память, ГБ: 6.6


In [ ]:
# Сначала сохранить, потом отключить. A100 тратит около 5,4 единицы в час, даже когда ничего не считает.
AUTO_OFF = True     # False — оставить сессию
import os
OUT = "/content/drive/MyDrive/vkr1_results" if os.path.isdir("/content/drive/MyDrive") else "."
path = f"{OUT}/quant_check.csv"
print("сохранено:", path, "|", f"{sum(1 for _ in open(path, encoding='utf-8')) - 1} строк" if os.path.exists(path) else "файла нет")
if AUTO_OFF:
    try:
        from google.colab import drive, runtime
        if os.path.isdir("/content/drive/MyDrive"):
            drive.flush_and_unmount()     # дождаться, пока файл уйдёт на Google Drive
        runtime.unassign()
    except ImportError:
        pass        # не Colab — отключать нечего
